# Notebook 04 — Carga a RDS

Los notebooks anteriores produjeron estas capas en S3:

| Notebook | Capa | Contenido |
|----------|------|-----------|
| 01 - ETL | Bronze / Silver | CSVs crudos y datos limpios de la flota |
| 02 - EDA + Features | Silver → Gold | Features engineered para el modelo |
| 03 - Modelado | Gold / Models | `risk_scores`, `model.pkl` y `model_metadata.json` |

Este notebook agrega la capa **RDS**.

## Arquitectura

In [1]:
from IPython.display import HTML

HTML("""
<pre class="mermaid" style="background:white; padding:16px;">
%%{init: {'theme':'base', 'themeVariables': {
    'primaryColor':'#e3f2fd','primaryTextColor':'#000',
    'primaryBorderColor':'#1565c0','lineColor':'#546e7a'
}}}%%
graph TB
    subgraph S3["Amazon S3 - Data Lake"]
        direction TB
        SLV["Silver\nvehicles · service"]
        RSK["Gold/risk_scores\noutput NB03"]
        MDL["models/\nmodel_metadata.json"]
    end

    subgraph RDS["Amazon RDS PostgreSQL (autoshop-rds)"]
        direction TB
        PRI["Primaria\nCRUD + carga inicial"]
        REP["Replica\nlecturas de Streamlit"]
        PRI -->|replicacion asincrona| REP
    end

    subgraph APP["Aplicacion"]
        STR["Streamlit\nVista Gerente"]
        AGT["Strands Agent\nVista Mecanico"]
    end

    SLV -->|Notebook 04| PRI
    RSK -->|Notebook 04| PRI
    MDL -->|Notebook 04| PRI
    REP -->|SELECT| STR
    AGT -->|INSERT maintenance_records| PRI

    classDef silver fill:#eceff1,stroke:#546e7a,color:#000
    classDef gold fill:#fff8e1,stroke:#f9a825,color:#000
    classDef pri fill:#e3f2fd,stroke:#1565c0,color:#000
    classDef rep fill:#e8f5e9,stroke:#2e7d32,color:#000
    classDef app fill:#f3e5f5,stroke:#7b1fa2,color:#000

    class SLV silver
    class RSK,MDL gold
    class PRI pri
    class REP rep
    class STR,AGT app
</pre>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
  mermaid.initialize({ startOnLoad: false });
  await mermaid.run();
</script>
""")

## Instalacion de dependencias

In [2]:
%pip install sqlalchemy psycopg2-binary awswrangler pyyaml -q

Note: you may need to restart the kernel to use updated packages.


---
## 1. Conexion a RDS

### Imports y configuracion

In [3]:
import hashlib
import json
import logging
import time
import warnings

import boto3
import awswrangler as wr
import pandas as pd
import yaml

from botocore.exceptions import ClientError
from sqlalchemy import (
    create_engine, text, insert, select,
    Integer, String, Float, Numeric, Boolean, DateTime, Text,
    ForeignKey,
)
from sqlalchemy.orm import (
    DeclarativeBase, Session, mapped_column, Mapped,
)
from sqlalchemy.exc import SQLAlchemyError
from typing import Optional

warnings.filterwarnings("ignore")

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s | %(levelname)s | %(message)s",
    datefmt = "%Y-%m-%d %H:%M:%S",
    force   = True,
)
log = logging.getLogger("notebook_04")

with open("../config.yaml") as f:
    CFG = yaml.safe_load(f)

BUCKET      = CFG["aws"]["s3_bucket"]
SILVER_PATH = f"s3://{BUCKET}/{CFG['aws']['s3_paths']['silver']}"
RSK_PATH    = f"s3://{BUCKET}/{CFG['aws']['s3_paths']['risk_scores']}"
MODEL_PATH  = f"s3://{BUCKET}/{CFG['aws']['s3_paths']['models']}"
SECRET_NAME = CFG["aws"]["secret_name"]
REGION      = CFG["aws"]["region"]

log.info("Config cargada — bucket: %s | region: %s", BUCKET, REGION)

2026-05-22 03:03:32 | INFO | Config cargada — bucket: itam-final-azucena | region: us-east-1


### Obtener credenciales desde Secrets Manager

In [4]:
def get_secret(secret_name: str, region_name: str = "us-east-1") -> dict:
    """Lee las credenciales del RDS desde AWS Secrets Manager."""
    session = boto3.session.Session()
    client  = session.client(service_name="secretsmanager", region_name=region_name)
    try:
        response = client.get_secret_value(SecretId=secret_name)
    except ClientError as e:
        log.error("Error al obtener el secret: %s", e)
        raise
    return json.loads(response["SecretString"])


creds = get_secret(SECRET_NAME, REGION)

RDS_ENDPOINT = CFG["rds"]["host"]
DB_NAME      = creds["dbname"]
DB_USER      = creds["username"]
DB_PASSWORD  = creds["password"]
DB_PORT      = creds["port"]

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{RDS_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)

try:
    with engine.connect() as conn:
        print("✓ Conexion exitosa a RDS PostgreSQL")
except SQLAlchemyError as e:
    print(f"✗ Error de conexion: {e}")

✓ Conexion exitosa a RDS PostgreSQL


---
## 2. Schema con SQLAlchemy ORM

In [5]:
class Base(DeclarativeBase):
    pass


class Vehicle(Base):
    """Catalogo de vehiculos. plate generado con SHA-256(vehicle_id)."""
    __tablename__ = "vehicles"
    vehicle_id:          Mapped[int]           = mapped_column(Integer, primary_key=True)
    plate:               Mapped[str]           = mapped_column(String(20), unique=True, nullable=False)
    make_and_model:      Mapped[Optional[str]] = mapped_column(String(100))
    year_of_manufacture: Mapped[Optional[int]] = mapped_column(Integer)
    vehicle_type:        Mapped[Optional[str]] = mapped_column(String(20))
    route_info:          Mapped[Optional[str]] = mapped_column(String(20))


class RiskScore(Base):
    """Scores de riesgo pre-computados por XGBoost (NB03)."""
    __tablename__ = "risk_scores"
    vehicle_id:           Mapped[int]   = mapped_column(Integer, ForeignKey("vehicles.vehicle_id"), primary_key=True)
    risk_score:           Mapped[float] = mapped_column(Float, nullable=False)
    maintenance_required: Mapped[bool]  = mapped_column(Boolean, nullable=False)
    risk_level:           Mapped[str]   = mapped_column(String(10), nullable=False)
    computed_at:          Mapped[str]   = mapped_column(DateTime, nullable=False)
    model_version:        Mapped[str]   = mapped_column(String(20), nullable=False)


class ServiceCatalog(Base):
    """Base de conocimiento de problemas y soluciones para el agente."""
    __tablename__ = "service_catalog"
    service_id:      Mapped[int]           = mapped_column(Integer, primary_key=True, autoincrement=True)
    common_problem:  Mapped[str]           = mapped_column(String(100), nullable=False)
    solution_used:   Mapped[str]           = mapped_column(String(200), nullable=False)
    vehicle_company: Mapped[Optional[str]] = mapped_column(String(50))


class MaintenanceRecord(Base):
    """Registros de servicio capturados por el agente."""
    __tablename__ = "maintenance_records"
    record_id:      Mapped[int]             = mapped_column(Integer, primary_key=True, autoincrement=True)
    vehicle_id:     Mapped[int]             = mapped_column(Integer, ForeignKey("vehicles.vehicle_id"), nullable=False)
    service_date:   Mapped[str]             = mapped_column(DateTime, nullable=False)
    common_problem: Mapped[Optional[str]]   = mapped_column(String(100))
    solution_used:  Mapped[Optional[str]]   = mapped_column(String(200))
    mechanic_notes: Mapped[Optional[str]]   = mapped_column(Text)
    cost:           Mapped[Optional[float]] = mapped_column(Numeric(10, 2))
    registered_at:  Mapped[Optional[str]]   = mapped_column(DateTime)


class ModelRunLog(Base):
    """Registro de ejecuciones del pipeline de modelado."""
    __tablename__ = "model_run_log"
    run_id:        Mapped[int]             = mapped_column(Integer, primary_key=True, autoincrement=True)
    model_version: Mapped[Optional[str]]   = mapped_column(String(20))
    trained_at:    Mapped[Optional[str]]   = mapped_column(DateTime)
    accuracy:      Mapped[Optional[float]] = mapped_column(Float)
    n_vehicles:    Mapped[Optional[int]]   = mapped_column(Integer)
    features_used: Mapped[Optional[str]]   = mapped_column(Text)


log.info("✓ 5 clases ORM definidas: vehicles, risk_scores, service_catalog, "
         "maintenance_records, model_run_log")

2026-05-22 03:04:25 | INFO | ✓ 5 clases ORM definidas: vehicles, risk_scores, service_catalog, maintenance_records, model_run_log


### Diagrama Entidad-Relacion (ERD)

In [6]:
HTML("""
<pre class="mermaid" style="background:white; padding:16px;">
erDiagram
    VEHICLES {
        int     vehicle_id          PK
        string  plate
        string  make_and_model
        int     year_of_manufacture
        string  vehicle_type
        string  route_info
    }
    RISK_SCORES {
        int     vehicle_id           PK,FK
        float   risk_score
        bool    maintenance_required
        string  risk_level
        datetime computed_at
        string  model_version
    }
    SERVICE_CATALOG {
        int     service_id      PK
        string  common_problem
        string  solution_used
        string  vehicle_company
    }
    MAINTENANCE_RECORDS {
        int     record_id      PK
        int     vehicle_id     FK
        datetime service_date
        string  common_problem
        string  solution_used
        text    mechanic_notes
        numeric cost
        datetime registered_at
    }
    MODEL_RUN_LOG {
        int     run_id        PK
        string  model_version
        datetime trained_at
        float   accuracy
        int     n_vehicles
        text    features_used
    }

    VEHICLES ||--o| RISK_SCORES         : "tiene score"
    VEHICLES ||--o{ MAINTENANCE_RECORDS : "tiene registros"
</pre>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
  mermaid.initialize({ startOnLoad: false });
  await mermaid.run();
</script>
""")

### Crear el schema en PostgreSQL

In [16]:
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename"
    ))
    tablas = [row[0] for row in result]

log.info("✓ Tablas en PostgreSQL: %s", tablas)

2026-05-22 03:17:00 | INFO | ✓ Tablas en PostgreSQL: ['maintenance_records', 'model_run_log', 'risk_scores', 'service_catalog', 'vehicles']


---
## 3. Carga de tablas desde Silver

In [17]:
def load_parquet(session, model, s3_path: str, columns: list) -> int:
    """Lee un Parquet desde S3 e inserta en RDS por chunks.

    Returns:
        Numero de filas insertadas.
    """
    df = wr.s3.read_parquet(path=s3_path, dataset=True, columns=columns)

    records = [
        {k: None if pd.isnull(v) else v for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]

    CHUNK = 5_000
    for i in range(0, len(records), CHUNK):
        session.execute(insert(model), records[i: i + CHUNK])

    log.info("✓ %s: %d filas cargadas", model.__tablename__, len(records))
    return len(records)

### Cargar vehicles

Se genera la placa sintetica.

In [18]:
def generate_plate(vehicle_id: int, salt: int = 0) -> str:
    """Genera placa sintética determinista: SHA-256(vehicle_id:salt) → XXX-000."""
    key = f"{vehicle_id}:{salt}"
    h   = int(hashlib.sha256(key.encode()).hexdigest(), 16)
    letters = "ABCDEFGHJKLMNPRSTUVWXYZ"
    l1     = letters[h % len(letters)]
    l2     = letters[(h >> 8)  % len(letters)]
    l3     = letters[(h >> 16) % len(letters)]
    digits = (h >> 24) % 1000
    return f"{l1}{l2}{l3}-{digits:03d}"


def assign_plates(vehicle_ids: pd.Series) -> list:
    """Retorna lista de placas únicas en el mismo orden que vehicle_ids."""
    used   = set()
    result = []
    for vid in vehicle_ids:
        salt  = 0
        plate = generate_plate(vid, salt)
        while plate in used:
            salt += 1
            plate = generate_plate(vid, salt)
        used.add(plate)
        result.append(plate)
    return result


t0 = time.time()
df_silver = wr.s3.read_parquet(path=f"{SILVER_PATH}/logistics/", dataset=True)
df_silver["plate"] = assign_plates(df_silver["Vehicle_ID"])

df_vehicles = (
    df_silver[["Vehicle_ID", "plate", "Make_and_Model",
               "Year_of_Manufacture", "Vehicle_Type", "Route_Info"]]
    .rename(columns={
        "Vehicle_ID":          "vehicle_id",
        "Make_and_Model":      "make_and_model",
        "Year_of_Manufacture": "year_of_manufacture",
        "Vehicle_Type":        "vehicle_type",
        "Route_Info":          "route_info",
    })
    .drop_duplicates(subset=["vehicle_id"])
)

records_vehicles = [
    {k: None if pd.isnull(v) else v for k, v in row.items()}
    for row in df_vehicles.to_dict(orient="records")
]

with Session(engine) as session:
    CHUNK = 5_000
    for i in range(0, len(records_vehicles), CHUNK):
        session.execute(insert(Vehicle), records_vehicles[i: i + CHUNK])
    session.commit()

log.info("✓ vehicles: %d filas cargadas (%.1fs)", len(df_vehicles), time.time() - t0)

2026-05-22 03:17:13 | INFO | ✓ vehicles: 92000 filas cargadas (5.6s)


### Cargar service_catalog

Base de conocimiento para el agente

In [19]:
t0 = time.time()
df_service = wr.s3.read_parquet(path=f"{SILVER_PATH}/service/", dataset=True)
df_catalog  = (
    df_service[["common_problem", "solution_used", "vehicle_company"]]
    .drop_duplicates(subset=["common_problem", "solution_used"])
    .reset_index(drop=True)
)
df_catalog.insert(0, "service_id", range(1, len(df_catalog) + 1))

records_catalog = [
    {k: None if pd.isnull(v) else v for k, v in row.items()}
    for row in df_catalog.to_dict(orient="records")
]

with Session(engine) as session:
    session.execute(insert(ServiceCatalog), records_catalog)
    session.commit()

log.info("✓ service_catalog: %d filas cargadas (%.1fs)", len(records_catalog), time.time() - t0)

2026-05-22 03:17:16 | INFO | ✓ service_catalog: 58 filas cargadas (0.4s)


---
## 4. Carga de tablas desde Gold

### Cargar risk_scores

Output de notebook 03 scores pre-computados para todos los vehiculos.


In [20]:
t0 = time.time()
df_risk = wr.s3.read_parquet(path=RSK_PATH, dataset=True)

assert len(df_risk) > 0, "Gold/risk_scores esta vacio — re-ejecuta NB03"
assert set(["vehicle_id","risk_score","maintenance_required",
            "risk_level","computed_at","model_version"]).issubset(df_risk.columns), \
    "Faltan columnas en risk_scores"

records_risk = [
    {k: None if pd.isnull(v) else v for k, v in row.items()}
    for row in df_risk.to_dict(orient="records")
]

with Session(engine) as session:
    CHUNK = 5_000
    for i in range(0, len(records_risk), CHUNK):
        session.execute(insert(RiskScore), records_risk[i: i + CHUNK])
    session.commit()

log.info("✓ risk_scores: %d filas cargadas (%.1fs)", len(df_risk), time.time() - t0)

2026-05-22 03:19:24 | INFO | ✓ risk_scores: 92000 filas cargadas (6.4s)


### Cargar model_run_log

Metadata de la ejecución

In [21]:
s3_client = boto3.client("s3")
obj = s3_client.get_object(
    Bucket = BUCKET,
    Key    = f"{CFG['aws']['s3_paths']['models']}/model_metadata.json",
)
metadata = json.loads(obj["Body"].read())

import pandas as _pd
with Session(engine) as session:
    session.execute(insert(ModelRunLog), [{
        "model_version": metadata["model_version"],
        "trained_at":    _pd.Timestamp(metadata["trained_at"]),
        "accuracy":      metadata["accuracy"],
        "n_vehicles":    metadata["n_vehicles"],
        "features_used": metadata["features_used"],
    }])
    session.commit()

log.info("✓ model_run_log: entrada registrada — version=%s | accuracy=%.4f",
         metadata["model_version"], metadata["accuracy"])

2026-05-22 03:19:46 | INFO | ✓ model_run_log: entrada registrada — version=20260522_0217 | accuracy=0.9998


---
## 5. Simulacion de operaciones CRUD en `maintenance_records`

Simulamos las operaciones para verificar que el schema funciona.

### Create — INSERT

Simula lo que hace el agente cuando el mecanico registra un servicio.

In [22]:
with Session(engine) as session:
    registro_prueba = MaintenanceRecord(
        vehicle_id     = 1,
        service_date   = pd.Timestamp.now(),
        common_problem = "Brake noise",
        solution_used  = "Brake pad replacement",
        mechanic_notes = "Desgaste uniforme en pastillas delanteras — reemplazo completo",
        cost           = 850.00,
        registered_at  = pd.Timestamp.now(),
    )
    session.add(registro_prueba)
    session.commit()
    log.info("✓ Create: maintenance_record id=%s creado", registro_prueba.record_id)

2026-05-22 03:20:05 | INFO | ✓ Create: maintenance_record id=1 creado


### Read — SELECT con JOIN

In [23]:
df_record_check = pd.read_sql(
    """
    SELECT mr.record_id, v.plate, v.make_and_model,
           mr.service_date, mr.common_problem, mr.solution_used, mr.cost
    FROM   maintenance_records mr
    JOIN   vehicles v ON mr.vehicle_id = v.vehicle_id
    ORDER  BY mr.registered_at DESC
    LIMIT  5
    """,
    engine,
)
log.info("Registros de mantenimiento:\n%s", df_record_check.to_string(index=False))

2026-05-22 03:20:11 | INFO | Registros de mantenimiento:
 record_id   plate make_and_model               service_date common_problem         solution_used  cost
         1 JEG-719     Ford F-150 2026-05-22 03:20:05.960667    Brake noise Brake pad replacement 850.0


### Delete — limpiar el registro de prueba

In [24]:
with Session(engine) as session:
    record_id_prueba = session.execute(
        text("SELECT record_id FROM maintenance_records WHERE common_problem = 'Brake noise' LIMIT 1")
    ).scalar()
    if record_id_prueba:
        session.execute(
            text("DELETE FROM maintenance_records WHERE record_id = :rid"),
            {"rid": record_id_prueba},
        )
        session.commit()
        log.info("✓ Delete: registro de prueba eliminado (id=%s)", record_id_prueba)

2026-05-22 03:20:14 | INFO | ✓ Delete: registro de prueba eliminado (id=1)


---
## 6. Verificacion de conteos en la RDS primaria

In [25]:
query_counts = """
SELECT 'vehicles'             AS tabla, COUNT(*) AS filas FROM vehicles
UNION ALL
SELECT 'risk_scores',                   COUNT(*)           FROM risk_scores
UNION ALL
SELECT 'service_catalog',               COUNT(*)           FROM service_catalog
UNION ALL
SELECT 'maintenance_records',           COUNT(*)           FROM maintenance_records
UNION ALL
SELECT 'model_run_log',                 COUNT(*)           FROM model_run_log
ORDER BY tabla
"""

df_counts = pd.read_sql(query_counts, engine)
log.info("Conteos en RDS primaria:\n%s", df_counts.to_string(index=False))

EXPECTED = {
    "vehicles":            (1,    None),
    "risk_scores":         (1,    None),
    "service_catalog":     (1,    None),
    "maintenance_records": (0,    0),
    "model_run_log":       (1,    None),
}

counts  = dict(zip(df_counts["tabla"], df_counts["filas"]))
all_ok  = True
for tabla, (lo, hi) in EXPECTED.items():
    n      = counts.get(tabla, 0)
    ok     = (n >= lo) and (hi is None or n <= hi)
    symbol = "✓" if ok else "❌"
    log.info("  %s %-25s %8d filas", symbol, tabla, n)
    if not ok:
        all_ok = False

assert all_ok, "Hay tablas con conteos inesperados — revisa los logs"
log.info("✓ Todas las tablas verificadas correctamente")

2026-05-22 03:20:18 | INFO | Conteos en RDS primaria:
              tabla  filas
maintenance_records      0
      model_run_log      1
        risk_scores  92000
    service_catalog     58
           vehicles  92000


2026-05-22 03:20:18 | INFO |   ✓ vehicles                     92000 filas


2026-05-22 03:20:18 | INFO |   ✓ risk_scores                  92000 filas


2026-05-22 03:20:18 | INFO |   ✓ service_catalog                 58 filas


2026-05-22 03:20:18 | INFO |   ✓ maintenance_records              0 filas


2026-05-22 03:20:18 | INFO |   ✓ model_run_log                    1 filas


2026-05-22 03:20:18 | INFO | ✓ Todas las tablas verificadas correctamente


---
## 7. Verificacion de la replica de lectura

In [26]:
RDS_REPLICA_ENDPOINT = CFG["rds"]["host_replica"]

if not RDS_REPLICA_ENDPOINT or RDS_REPLICA_ENDPOINT == "TBD":
    log.info("Sin replica configurada — actualiza rds.host_replica en config.yaml")
else:
    try:
        engine_replica = create_engine(
            f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
            f"@{RDS_REPLICA_ENDPOINT}:{DB_PORT}/{DB_NAME}"
        )
        with engine_replica.connect() as conn:
            n_replica  = conn.execute(text("SELECT COUNT(*) FROM vehicles")).scalar()
        n_primaria = counts["vehicles"]

        log.info("Replica — vehicles primaria: %d | replica: %d", n_primaria, n_replica)

        if n_replica < n_primaria:
            log.warning("Replica aun sincronizando (%d/%d) — espera ~10s y re-ejecuta",
                        n_replica, n_primaria)
        else:
            log.info("✓ Replica sincronizada correctamente")

    except Exception as e:
        log.warning("No se pudo verificar la replica: %s", e)

2026-05-22 03:20:24 | INFO | Replica — vehicles primaria: 92000 | replica: 92000


2026-05-22 03:20:24 | INFO | ✓ Replica sincronizada correctamente
